In [11]:
import pandas as pd
from surprise import Dataset, Reader, SVD, SVDpp
from surprise.model_selection import train_test_split
from surprise import accuracy
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
import numpy as np
from sklearn.metrics import ndcg_score

In [2]:
import pandas as pd

# Load datasets
ratings = pd.read_csv("./MovieLensDataset/rating.csv")
movies = pd.read_csv("./MovieLensDataset/movie.csv")


In [4]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [16]:
ratings = ratings.drop('timestamp', axis=1)
ratings.head()

,userId,movieId,rating
0,1,2,3.5
1,1,29,3.5
2,1,32,3.5
3,1,47,3.5
4,1,50,3.5


In [ ]:
movies.isnull().sum()

userId     0
movieId    0
rating     0
dtype: int64

In [ ]:
ratings.isnull().sum()

userId     0
movieId    0
rating     0
dtype: int64

In [ ]:
#movies.shape
ratings.shape

(20000263, 3)

In [21]:
# 3. Sparse Matrix (for Item-based CF)
# =========================
# Map userId and movieId to indices
user_map = {u: i for i, u in enumerate(ratings['userId'].unique())}
movie_map = {m: i for i, m in enumerate(ratings['movieId'].unique())}
user_inv_map = {i: u for u, i in user_map.items()}
movie_inv_map = {i: m for m, i in movie_map.items()}

rows = ratings['userId'].map(user_map)
cols = ratings['movieId'].map(movie_map)
vals = ratings['rating']
user_item_sparse = csr_matrix((vals, (rows, cols)), 
                              shape=(len(user_map), len(movie_map)))

print("Sparse matrix shape:", user_item_sparse.shape)


Sparse matrix shape: (138493, 26744)


In [6]:
# 4. Item-based Collaborative Filtering
# =========================
item_similarity = cosine_similarity(user_item_sparse.T, dense_output=False)

def recommend_movies_cf(user_id, N=10):
    if user_id not in user_map:
        return pd.DataFrame({"Error": ["User not found"]})
    
    user_idx = user_map[user_id]
    user_ratings = user_item_sparse[user_idx].toarray().ravel()
    rated_movies = np.where(user_ratings > 0)[0]
    
    scores = {}
    for m in rated_movies:
        sim_scores = item_similarity[m].toarray().ravel()
        for idx, score in enumerate(sim_scores):
            if idx not in rated_movies and score > 0:
                scores[idx] = scores.get(idx, 0) + score * user_ratings[m]
    
    top_movies_idx = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:N]
    top_movie_ids = [movie_inv_map[idx] for idx, _ in top_movies_idx]
    
    return movies[movies["movieId"].isin(top_movie_ids)][["title", "genres"]]


In [ ]:
# 5. Matrix Factorization (SVD – Surprise)
# =========================
from surprise.model_selection import GridSearchCV

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2)


param_grid = {
    'n_factors': [50, 100, 150],
    'n_epochs': [20, 40, 60],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.02, 0.05]
}
svd = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3)
svd.fit(data)

print("Best RMSE:", svd.best_score['rmse'])
print("Best parameters:", svd.best_params['rmse'])

predictions = svd.test(testset)
print("RMSE:", accuracy.rmse(predictions))

def recommend_movies(user_id, N=10):
    if user_id not in ratings['userId'].unique():
        return pd.DataFrame({"Error": ["User not found"]})
    
    movie_ids = movies["movieId"].unique()
    user_movies = ratings[ratings["userId"] == user_id]["movieId"].tolist()
    
    preds = [(mid, svd.predict(user_id, mid).est) for mid in movie_ids if mid not in user_movies]
    preds.sort(key=lambda x: x[1], reverse=True)
    top_movies = [mid for mid, _ in preds[:N]]
    
    return movies[movies["movieId"].isin(top_movies)][["title", "genres"]]


RMSE: 0.7856
RMSE: 0.7856112474153825


In [ ]:
# 6. Evaluation Metrics
# =========================
def precision_recall_at_k(predictions, k=10, threshold=3.5):
    user_est_true = {}
    for uid, _, true_r, est, _ in predictions:
        user_est_true.setdefault(uid, []).append((est, true_r))

    precisions, recalls = [], []
    for uid, ratings in user_est_true.items():
        ratings.sort(key=lambda x: x[0], reverse=True)
        top_k = ratings[:k]
        n_rel = sum((true_r >= threshold) for (_, true_r) in ratings)
        n_rec_k = sum((est >= threshold) for (est, true_r) in top_k)
        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold)) for (est, true_r) in top_k)

        precisions.append(n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0)
        recalls.append(n_rel_and_rec_k / n_rel if n_rel != 0 else 0)

    return sum(precisions) / len(precisions), sum(recalls) / len(recalls)

prec, rec = precision_recall_at_k(predictions, k=10)
print("Precision@10:", prec)
print("Recall@10:", rec)

# NDCG Example
# Build user_est_true dictionary outside
user_est_true = {}
for uid, _, true_r, est, _ in predictions:
    user_est_true.setdefault(uid, []).append((est, true_r))

# Example: pick one user
sample_uid = list(user_est_true.keys())[0]

# Sort predictions for that user
user_est_true[sample_uid].sort(key=lambda x: x[0], reverse=True)

# Extract top-10 true ratings & predicted scores
y_true = [[true_r for (_, true_r) in user_est_true[sample_uid][:10]]]
y_score = [[est for (est, _) in user_est_true[sample_uid][:10]]]

print("NDCG@10:", ndcg_score(y_true, y_score))


Precision@10: 0.7764229082721056
Recall@10: 0.5647736372446464
NDCG@10: 0.9762743697403702


In [ ]:
# 7. Example Usage
# =========================
print("\nCF Recommendations for user 1:")
print(recommend_movies_cf(1, 5))

print("\nSVD Recommendations for user 1:")
print(recommend_movies(1, 5))


CF Recommendations for user 1:
                                                  title  \
1184  Star Wars: Episode VI - Return of the Jedi (1983)   
1237                               Groundhog Day (1993)   
1242                          Back to the Future (1985)   
1528                   Men in Black (a.k.a. MIB) (1997)   
2486                                 Matrix, The (1999)   

                       genres  
1184  Action|Adventure|Sci-Fi  
1237   Comedy|Fantasy|Romance  
1242  Adventure|Comedy|Sci-Fi  
1528     Action|Comedy|Sci-Fi  
2486   Action|Sci-Fi|Thriller  

SVD Recommendations for user 1:
                                                   title               genres
8615   Crooks in Clover (a.k.a. Monsieur Gangster) (L...  Action|Comedy|Crime
19337                                 Bleak House (2005)                Drama
20823           Death on the Staircase (Soupçons) (2004)    Crime|Documentary
21766                    Century of the Self, The (2002)          Documentar

In [ ]:
import joblib

# Save model
joblib.dump(svd, "svdpp_model_7866.pkl")
print("The model is saved...")

The model is saved...
